# ATC Radar - ASTERIX Decoder
## Visualización interactiva de datos radar

Este notebook carga archivos PCAP y visualiza los datos ASTERIX como lo hace la aplicación PyQt.

In [ ]:
import sys
import os
sys.path.insert(0, '/mnt/c/documentos/decode_asterix')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import time
from collections import defaultdict

# Importar módulos de la aplicación
from io_tools import load_pcap, load_ast
from track_manager import TrackManager, TargetProcessor, SensorRegistry
from decoders import AsterixRecord

print('✓ Módulos importados correctamente')

## 1. Cargar archivo PCAP

In [ ]:
# Seleccionar archivo PCAP
pcap_files = list(Path('/mnt/c/documentos/decode_asterix').glob('*.pcap'))
print(f'Archivos PCAP disponibles ({len(pcap_files)}):' )
for i, f in enumerate(pcap_files[:10]):
    print(f'  {i+1}. {f.name} ({f.stat().st_size / 1024 / 1024:.2f} MB)')

# Usar el primer archivo
if pcap_files:
    selected_file = pcap_files[0]
    print(f'\nCargando: {selected_file.name}')
    
    try:
        raw_data = load_pcap(str(selected_file))
        print(f'✓ Datos cargados: {len(raw_data)} bytes')
    except Exception as e:
        print(f'✗ Error: {e}')
else:
    print('No hay archivos PCAP disponibles')

## 2. Decodificar datos ASTERIX

In [ ]:
import asterix

print('Decodificando datos ASTERIX...')
records = asterix.parse(raw_data)
print(f'✓ {len(records)} registros decodificados')

# Mostrar primeros registros
print('\nPrimeros 3 registros:')
for i, rec in enumerate(records[:3]):
    print(f'\nRegistro {i+1}:')
    print(f'  Categoría: CAT {rec.get("category", "N/A")}')
    print(f'  Timestamp: {rec.get("timestamp", "N/A")}')
    print(f'  Azimuth: {rec.get("azimuth", "N/A")}')
    print(f'  Range: {rec.get("range", "N/A")}')

## 3. Visualizar datos en mapa polar (como la GUI)

In [ ]:
# Extraer datos de coordenadas polares
azimuths = []
ranges = []
categories = []

for rec in records:
    if 'I048/040' in rec or 'I034/020' in rec:
        az_data = rec.get('I048/040') or rec.get('I034/020')
        if isinstance(az_data, dict):
            az = az_data.get('THETA') or az_data.get('AZM')
            rng_data = rec.get('I048/040') or rec.get('I034/020')
            rng = rng_data.get('RHO') or rng_data.get('DIST')
            
            if az is not None and rng is not None:
                azimuths.append(float(az))
                ranges.append(float(rng))
                categories.append(rec.get('category', 0))

print(f'✓ {len(azimuths)} puntos extraídos')

# Crear gráfico polar como la GUI
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='polar')

# Convertir azimuth de grados a radianes
azimuths_rad = np.array(azimuths) * np.pi / 180
ranges_nm = np.array(ranges)  # Ya en NM

# Plot por categoría
for cat in set(categories):
    mask = np.array(categories) == cat
    ax.scatter(azimuths_rad[mask], ranges_nm[mask], 
              label=f'CAT {cat}', s=20, alpha=0.6)

ax.set_ylim(0, max(ranges_nm) if ranges_nm.size > 0 else 250)
ax.set_title('Radar Táctico - Vista Polar', pad=20, fontsize=14)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.show()

print('✓ Gráfico generado')

## 4. Estadísticas

In [ ]:
# Agrupar por categoría
stats = defaultdict(int)
for rec in records:
    cat = rec.get('category', 'Unknown')
    stats[f'CAT {cat}'] += 1

print('Distribución por categoría ASTERIX:')
print('=' * 40)
for cat, count in sorted(stats.items()):
    print(f'{cat:15} : {count:6d} registros')

print('\nTotal de registros:', sum(stats.values()))

# Gráfico de barras
fig, ax = plt.subplots(figsize=(10, 5))
cats = list(stats.keys())
counts = list(stats.values())
ax.bar(cats, counts, color='#00ff00', edgecolor='#00ffff', linewidth=2)
ax.set_ylabel('Número de registros', fontsize=12, color='#00ff00')
ax.set_title('Distribución de Categorías ASTERIX', fontsize=14, color='#00ffff')
ax.set_facecolor('#1a1a1a')
fig.patch.set_facecolor('#1a1a1a')
for text in ax.get_xticklabels() + ax.get_yticklabels():
    text.set_color('#00ff00')
for spine in ax.spines.values():
    spine.set_color('#00ff00')
plt.tight_layout()
plt.show()

## 5. Crear tabla de datos

In [ ]:
# Convertir a DataFrame
data = []
for rec in records[:100]:  # Primeros 100
    data.append({
        'Category': f"CAT {rec.get('category', 'N/A')}",
        'Timestamp': rec.get('timestamp', 'N/A'),
        'Azimuth (°)': rec.get('I048/040', {}).get('THETA', 'N/A') if isinstance(rec.get('I048/040'), dict) else 'N/A',
        'Range (NM)': rec.get('I048/040', {}).get('RHO', 'N/A') if isinstance(rec.get('I048/040'), dict) else 'N/A',
        'Callsign': rec.get('I240', {}).get('Callsign', 'N/A') if isinstance(rec.get('I240'), dict) else 'N/A',
        'Squawk': rec.get('I070', {}).get('Reply', 'N/A') if isinstance(rec.get('I070'), dict) else 'N/A',
    })

df = pd.DataFrame(data)
print('Primeros 20 registros:')
print(df.head(20).to_string())
print(f'\nTotal en tabla: {len(df)} registros')